In [1]:
suppressPackageStartupMessages({
  library(Seurat)
  library(SeuratObject)
  library(dplyr)
  library(ggplot2)
  library(patchwork)
  library(Matrix)
  library(data.table)
})

start_time <- Sys.time()

DATASET <- "scMixology"
VERSION <- "template"
SEED <- 1234
set.seed(SEED)

BASE_DIR <- "/home/mtarzi/CEPH/scRNA_sequencing_scMixology_benchmarking"
OUT_DIR <- file.path(BASE_DIR, "seurat_outputs")
CELLRANGER_OUTS <- file.path(BASE_DIR, "cell_ranger", "SCMIXOLOGY", "outs")
MATRIX_DIR <- file.path(CELLRANGER_OUTS, "filtered_feature_bc_matrix")

TABLE_DIR <- file.path(OUT_DIR, "tables")
FIGURE_DIR <- file.path(OUT_DIR, "figures")
OBJECT_DIR <- file.path(OUT_DIR, "objects")

dir.create(TABLE_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(FIGURE_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(OBJECT_DIR, recursive = TRUE, showWarnings = FALSE)


In [2]:
counts <- Read10X(data.dir = MATRIX_DIR)

if (is.list(counts)) {
  if ("Gene Expression" %in% names(counts)) {
    counts <- counts[["Gene Expression"]]
  } else {
    counts <- counts[[1]]
  }
}

data_obj <- CreateSeuratObject(
  counts = counts,
  project = DATASET,
  min.cells = 0,
  min.features = 0
)

data_obj$sample <- "SCMIXOLOGY"
data_obj$barcode_cellranger <- colnames(data_obj)
data_obj$barcode_clean <- gsub("-1$", "", colnames(data_obj))
data_obj[["percent.mt"]] <- PercentageFeatureSet(data_obj, pattern = "^MT-|^mt-")
data_obj[["percent.rps"]] <- PercentageFeatureSet(data_obj, pattern = "^RPS|^Rps|^rps")
data_obj[["percent.rpl"]] <- PercentageFeatureSet(data_obj, pattern = "^RPL|^Rpl|^rpl")

data_obj


An object of class Seurat 
38606 features across 4488 samples within 1 assay 
Active assay: RNA (38606 features, 0 variable features)
 1 layer present: counts

In [3]:
METADATA_DIR <- "/CEPH/users/mtarzi/data/scMixology/metadata"
GENE_COUNT_FILE <- file.path(METADATA_DIR, "GSM3618014_gene_count.csv.gz")
BARCODE_FILE <- file.path(METADATA_DIR, "GSE126906_barcodes.csv.gz")
LABEL_FILE <- file.path(METADATA_DIR, "scmixology_labels_by_libid.csv")

input_files <- c(GENE_COUNT_FILE, BARCODE_FILE, LABEL_FILE)
missing_files <- input_files[!file.exists(input_files)]
if (length(missing_files) > 0) {
  stop(paste("Missing metadata files:", paste(missing_files, collapse = "; ")))
}


In [4]:
gene_count_header <- read.csv(
  gzfile(GENE_COUNT_FILE),
  nrows = 0,
  check.names = FALSE
)

lib_ids <- data.frame(
  lib_id = as.character(colnames(gene_count_header)[-1]),
  stringsAsFactors = FALSE
)

raw_barcodes <- read.csv(
  gzfile(BARCODE_FILE),
  header = FALSE,
  stringsAsFactors = FALSE
)

barcode_values <- as.character(unlist(raw_barcodes, use.names = FALSE))
barcode_16bp <- regmatches(barcode_values, regexpr("[ACGT]{16}", barcode_values))
barcode_16bp <- barcode_16bp[!is.na(barcode_16bp)]
barcode_16bp <- barcode_16bp[barcode_16bp != ""]

barcode_16bp <- data.frame(
  barcode_16bp = barcode_16bp,
  stringsAsFactors = FALSE
)

if (nrow(lib_ids) != nrow(barcode_16bp)) {
  stop(paste("Lib90 ID count and 16 bp barcode count do not match:", nrow(lib_ids), nrow(barcode_16bp)))
}

barcode_bridge <- data.frame(
  lib_id = lib_ids$lib_id,
  barcode_16bp = barcode_16bp$barcode_16bp,
  stringsAsFactors = FALSE
)

write.csv(
  barcode_bridge,
  file.path(TABLE_DIR, paste0(DATASET, "_seurat_", VERSION, "_Lib90_barcode_bridge.csv")),
  row.names = FALSE
)

barcode_bridge


lib_id,barcode_16bp
<chr>,<chr>
Lib90_00000,CACACAAAGCTAAGAT
Lib90_00001,TCAACGAGTTACGTCA
Lib90_00002,GACTACAAGGGCTCTC
Lib90_00003,GAAATGACACTCGACG
Lib90_00004,CACATAGTCCGTCATC
Lib90_00005,GACCTGGGTAGCTTGT
Lib90_00006,CACACCTTCCGTAGGC
Lib90_00007,CAGATCAAGTCACGCC
Lib90_00008,TAGACCATCCTTCAAT


In [5]:
reference_labels <- fread(LABEL_FILE, data.table = FALSE)

if (!all(c("lib_id", "cell_line") %in% colnames(reference_labels))) {
  stop("LABEL_FILE must contain columns named lib_id and cell_line")
}

reference_labels$lib_id <- as.character(reference_labels$lib_id)
reference_labels$cell_line <- as.character(reference_labels$cell_line)

reference_metadata <- merge(
  reference_labels,
  barcode_bridge,
  by = "lib_id",
  all.x = TRUE
)

if (any(duplicated(reference_metadata$barcode_16bp[!is.na(reference_metadata$barcode_16bp)]))) {
  stop("Duplicated 16 bp barcodes found in reference metadata")
}

write.csv(
  reference_metadata,
  file.path(TABLE_DIR, paste0(DATASET, "_seurat_", VERSION, "_reference_metadata_with_barcodes.csv")),
  row.names = FALSE
)

reference_metadata


lib_id,cell_line,barcode_16bp
<chr>,<chr>,<chr>
Lib90_00000,HCC827,CACACAAAGCTAAGAT
Lib90_00001,HCC827,TCAACGAGTTACGTCA
Lib90_00002,H838,GACTACAAGGGCTCTC
Lib90_00003,HCC827,GAAATGACACTCGACG
Lib90_00004,HCC827,CACATAGTCCGTCATC
Lib90_00005,HCC827,GACCTGGGTAGCTTGT
Lib90_00006,H1975,CACACCTTCCGTAGGC
Lib90_00007,H1975,CAGATCAAGTCACGCC
Lib90_00008,HCC827,TAGACCATCCTTCAAT


In [6]:
data_obj$barcode_16bp <- regmatches(
  data_obj$barcode_cellranger,
  regexpr("[ACGT]{16}", data_obj$barcode_cellranger)
)

label_map <- reference_metadata$cell_line
names(label_map) <- reference_metadata$barcode_16bp

data_obj$known_label <- unname(label_map[data_obj$barcode_16bp])
data_obj$known_label_plot <- ifelse(is.na(data_obj$known_label), "unmatched", data_obj$known_label)
data_obj$known_label <- as.factor(data_obj$known_label)
data_obj$known_label_plot <- as.factor(data_obj$known_label_plot)

metadata_matching_summary <- data.frame(
  dataset = DATASET,
  version = VERSION,
  n_cells = ncol(data_obj),
  n_genes = nrow(data_obj),
  n_reference_labelled_cells = length(unique(na.omit(reference_metadata$barcode_16bp))),
  n_cells_with_known_label = sum(!is.na(data_obj$known_label)),
  n_cells_without_known_label = sum(is.na(data_obj$known_label)),
  percent_cells_labelled = round(100 * sum(!is.na(data_obj$known_label)) / ncol(data_obj), 2)
)

write.csv(
  metadata_matching_summary,
  file.path(TABLE_DIR, paste0(DATASET, "_seurat_", VERSION, "_metadata_matching_summary.csv")),
  row.names = FALSE
)

metadata_matching_summary


dataset,version,n_cells,n_genes,n_reference_labelled_cells,n_cells_with_known_label,n_cells_without_known_label,percent_cells_labelled
<chr>,<chr>,<dbl>,<dbl>,<int>,<int>,<int>,<dbl>
scMixology,template,4488,38606,3918,3916,572,87.25


In [7]:
prefilter_object <- file.path(OBJECT_DIR, paste0(DATASET, "_seurat_prefilter_with_cell_line_labels.rds"))
saveRDS(data_obj, prefilter_object)

important_packages <- c("Seurat", "SeuratObject", "Matrix", "dplyr", "ggplot2", "patchwork", "data.table")
package_versions <- data.frame(
  package = important_packages,
  version = sapply(important_packages, function(pkg) {
    if (requireNamespace(pkg, quietly = TRUE)) as.character(packageVersion(pkg)) else "not installed"
  }),
  row.names = NULL
)

write.csv(
  package_versions,
  file.path(TABLE_DIR, paste0(DATASET, "_seurat_", VERSION, "_package_versions.csv")),
  row.names = FALSE
)

sink(file.path(TABLE_DIR, paste0(DATASET, "_seurat_", VERSION, "_sessionInfo.txt")))
sessionInfo()
sink()

total_runtime_seconds <- as.numeric(difftime(Sys.time(), start_time, units = "secs"))
total_runtime <- data.frame(
  dataset = DATASET,
  version = VERSION,
  total_runtime_seconds = total_runtime_seconds,
  total_runtime_minutes = total_runtime_seconds / 60,
  final_object_file = prefilter_object
)

write.csv(
  total_runtime,
  file.path(TABLE_DIR, paste0(DATASET, "_seurat_", VERSION, "_total_runtime.csv")),
  row.names = FALSE
)

total_runtime


R version 4.3.3 (2024-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Ubuntu 19.04

Matrix products: default
BLAS/LAPACK: /15K/home-dist/home/mtarzi/.conda/envs/scRNA-R/lib/libopenblasp-r0.3.34.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Europe/Madrid
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] data.table_1.17.8  Matrix_1.6-5       patchwork_1.3.2    ggplot2_3.5.2     
[5] dplyr_1.1.4        Seurat_5.3.0       SeuratObject_5.2.0 sp_2.2-0          

loaded via a namespace (and not attached):
  [1] deldir_2.

dataset,version,total_runtime_seconds,total_runtime_minutes,final_object_file
<chr>,<chr>,<dbl>,<dbl>,<chr>
scMixology,template,78.01738,1.30029,/home/mtarzi/CEPH/scRNA_sequencing_scMixology_benchmarking/scanpy_outputs/objects/scMixology_seurat_prefilter_with_cell_line_labels.rds
